# Chapter 08: Rows for the link chart

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "08"
__file__ = os.path.abspath("run.py")

## The scene

Okafor's office wall is covered in paper: vendors, bank accounts, names, arrows. He wants it in a
spreadsheet by morning. "Eight invoices from Reyes's mailbox," he says. "Every field, every
number, same shape every time. If one amount comes back as a sentence, the chart is useless."

The invoices are plain text files in [`case/invoices/`](../../case/invoices/). Malhotra's approved
travel is in [`case/malhotra-travel.csv`](../../case/malhotra-travel.csv).

## The concept

A model produces text. A program needs data. **Structured output** is everything you do to close
that gap, and there are three levels of it:

| Level | How | Guarantee |
|---|---|---|
| Ask nicely | "Return JSON with keys a, b, c." | None. Usually works, sometimes wraps it in prose or a code fence. |
| JSON mode | `format: "json"` | It will parse. Keys and types can still be anything. |
| Schema-constrained | `format: <JSON schema>` | It parses **and** matches the schema: required keys, types, enums. |

On top of all three you still need **validation** (does it match your model in code?) and
**verification** (is it true, compared with the source?). The schema guarantees shape. Nothing
guarantees truth.

```mermaid
flowchart LR
    I[Invoice text] --> P[Prompt + schema]
    P --> M[Model with<br/>constrained decoding]
    M --> J[JSON string]
    J --> V{Pydantic<br/>validates?}
    V -- no --> R[Retry with the<br/>validation error] --> M
    V -- yes --> C{Matches the<br/>source document?}
    C -- no --> H[Correct or flag<br/>for a human]
    C -- yes --> T[Row in the table]
    H --> T
    T --> X[Join with travel records<br/>in ordinary Python]
```

## How it works

**Constrained decoding.** When you pass a JSON schema as `format`, Ollama converts it into a
grammar. At every step, before the sampler picks a token, every token that would break the grammar
gets probability zero. The model cannot write a code fence, cannot put a string where a number
goes, and cannot invent a category outside the enum. It can still choose the wrong *allowed* value.

![Constrained decoding](../../docs/diagrams/08-constrained-decoding.svg)

The schema comes from a Pydantic model, so the same class both instructs the model and validates
the result:

```python
class Invoice(BaseModel):
    invoice_id: str = Field(description="The invoice number exactly as printed, e.g. GTS-1182")
    vendor: str = Field(description="Supplier company name, in normal capitalisation")
    invoice_date: date = Field(description="Invoice date as YYYY-MM-DD")
    category: Literal["fuel", "marine services", "equipment hire", "it", "cleaning",
                      "freight coordination", "other"]
    service: str
    total_kbd: float = Field(description="Total due as a plain number, no commas or currency")
    vat_number: str | None = Field(description="VAT registration, or null if none is printed")
    remit_to: str

r = chat(f"Extract this invoice.\n\n{text}", format=Invoice.model_json_schema(), temperature=0)
invoice = Invoice.model_validate_json(r.text)
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json
import time
from datetime import date
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

from lantern import case, chat, learn
from lantern.ui import (
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    reply,
    save_shot,
    step,
    your_turn,
)

chapter("08", "Rows for the link chart", """
Okafor's office wall is covered in paper: vendors, bank accounts, names, arrows. He wants it in a
spreadsheet by morning. "Eight invoices from Reyes's mailbox," he says. "Every field, every
number, same shape every time. If one amount comes back as a sentence, the chart is useless."
""")


class Invoice(BaseModel):
    """One supplier invoice. Field order matters: the model fills them top to bottom."""

    invoice_id: str = Field(description="The invoice number exactly as printed, e.g. GTS-1182")
    vendor: str = Field(description="Supplier company name, in normal capitalisation")
    invoice_date: date = Field(description="Invoice date as YYYY-MM-DD")
    category: Literal["fuel", "marine services", "equipment hire", "it", "cleaning",
                      "freight coordination", "other"]
    service: str = Field(description="What was supplied, copied from the description line")
    total_kbd: float = Field(description="Total due as a plain number, no commas or currency")
    vat_number: str | None = Field(description="VAT registration, or null if none is printed")
    remit_to: str = Field(description="Bank account and bank name for payment")


invoices = case.files("invoices")
sample = case.read("invoices/NW-0419.txt")
evidence("invoices/NW-0419.txt", sample)

# With --mine, extraction is yours: practice/exercise.py extract(), with its own retry and cap.
MINE = learn.mine()

p_nice = learn.predict("Asked nicely for JSON, with no schema: will the reply pass validation?",
                       ["yes", "no"])

### 1. Asking nicely: 'Return JSON'

In [ ]:
step("1. Asking nicely: 'Return JSON'")
ask = f"""Extract the invoice fields as JSON with keys invoice_id, vendor, invoice_date,
category, service, total_kbd, vat_number, remit_to.

{sample}"""
r = chat(ask, temperature=0, max_tokens=400)
reply("LANTERN, no schema", "```text\n" + r.text + "\n```",
      meta=f"{r.output_tokens} tokens")
try:
    Invoice.model_validate_json(r.text)
    no_schema_valid = True
    note("It parsed and validated this time. Asking nicely sometimes works.")
except ValidationError as e:
    no_schema_valid = False
    note(f"Failed validation with {e.error_count()} error(s). First: {e.errors()[0]['msg']}")
    note("Typical causes: a ```json fence, '94,000.00' as a string, a made-up category.")
learn.reveal(p_nice, answer=0 if no_schema_valid else 1,
             why="Without a schema the model writes what JSON usually looks like in its training "
                 "data: a friendly sentence, a fence, numbers with commas. Nothing stops it.")

### 2. Constrained decoding: pass the schema as `format`

In [ ]:
step("2. Constrained decoding: pass the schema as `format`")
schema = Invoice.model_json_schema()
reply("Schema sent to Ollama (abridged)", "```json\n" + json.dumps(
    {k: schema["properties"][k] for k in ("category", "total_kbd", "vat_number")},
    indent=2) + "\n```", style="blue")
note("Ollama turns the schema into a grammar. Tokens that would break it get zero probability.")


def extract(text: str, retries: int = 2) -> tuple[Invoice | None, int]:
    """Constrained extraction with validation. On failure, show the model its error and retry."""
    messages = [{"role": "user", "content": f"Extract this invoice.\n\n{text}"}]
    for attempt in range(1, retries + 2):
        r = chat(messages, format=schema, temperature=0, max_tokens=400)
        try:
            return Invoice.model_validate_json(r.text), attempt
        except ValidationError as e:
            messages += [{"role": "assistant", "content": r.text},
                         {"role": "user", "content": f"That failed validation: {e}. Fix it."}]
    return None, retries + 1


p_calls = learn.predict(f"With the schema and up to two retries each, how many model calls for "
                        f"{len(invoices)} invoices?",
                        [f"{len(invoices)}: every one valid first time",
                         f"{len(invoices) + 1} to {len(invoices) + 4}",
                         f"more than {len(invoices) + 4}"])
if MINE:
    def extract(text: str) -> tuple[Invoice | None, int]:  # noqa: F811  (yours, with --mine)
        return MINE.extract(text, chat=chat, model_cls=Invoice, max_attempts=3)

rows: list[Invoice] = []
start = time.perf_counter()
attempts_used = 0
for path in invoices:
    inv, attempts = extract(path.read_text())
    attempts_used += attempts
    if inv:
        rows.append(inv)
    else:
        note(f"{path.name}: still invalid after retries, flagged for a human")
note(f"{len(rows)}/{len(invoices)} invoices valid, {attempts_used} calls in total, "
     f"{time.perf_counter() - start:.0f}s")

t = Table(title="Extracted invoices (validated by Pydantic)")
for col in ("invoice_id", "vendor", "date", "category", "total_kbd", "vat"):
    t.add_column(col, justify="right" if col == "total_kbd" else "left")
for inv in rows:
    t.add_row(inv.invoice_id, inv.vendor, str(inv.invoice_date), inv.category,
              f"{inv.total_kbd:,.2f}", inv.vat_number or "[red]none[/red]")
console.print(t)
learn.reveal(p_calls, answer=learn.bucket(attempts_used, [len(invoices), len(invoices) + 4]),
             why=f"{attempts_used} calls. The schema becomes a grammar (or the provider's native "
                 "structured output), so a token that would break it cannot be chosen: shape "
                 "failures mostly disappear, and the retry is left for rules a grammar cannot "
                 "express.")

p_source = learn.predict("Every row passed validation. How many fields will the check against "
                         "the printed invoices flag?", ["none", "1 to 3", "4 or more"])

### 3. Check the machine against the paper

In [ ]:
step("3. Check the machine against the paper")
problems = 0
for inv in rows:
    printed = next(p for p in invoices if p.stem == inv.invoice_id).read_text()
    if f"{inv.total_kbd:,.2f}" not in printed:
        problems += 1
        note(f"{inv.invoice_id}: total {inv.total_kbd:,.2f} does not appear on the invoice")
    vat_line = next((ln for ln in printed.splitlines() if ln.startswith("VAT registration:")), "")
    if inv.vat_number and inv.vat_number not in vat_line:
        problems += 1
        note(f"{inv.invoice_id}: vat_number '{inv.vat_number}' is not a VAT number on the invoice "
             "(there is no VAT line). Corrected to null.")
        inv.vat_number = None
note(f"{problems} field(s) contradicted the source documents out of {len(rows)} invoices checked.")
note("Valid JSON is not the same as correct JSON. The schema guarantees shape, not truth.")
note("Categories need a human eye too: is a network switch 'it' or 'equipment hire'?")
learn.reveal(p_source, answer=learn.bucket(problems, [0, 3]),
             why="A schema guarantees the shape and the types. It cannot know that a bank "
                 "account is not a VAT number; only a check against the source can.")

p_trips = learn.predict("How many of the Northwind invoices fall inside one of Malhotra's trips?",
                        ["none", "some", "all of them"])

### 4. Cross the Northwind rows with Malhotra's travel

In [ ]:
step("4. Cross the Northwind rows with Malhotra's travel")
trips = case.rows("malhotra-travel.csv")
nw = [inv for inv in rows if "northwind" in inv.vendor.lower()]
x = Table(title="Northwind invoices against R. Malhotra's approved travel")
for col in ("invoice", "date", "amount", "Malhotra was in", "trip"):
    x.add_column(col)
in_trips = 0
for inv in nw:
    trip = next((tr for tr in trips if date.fromisoformat(tr["depart"]) <= inv.invoice_date
                 <= date.fromisoformat(tr["return"])), None)
    in_trips += trip is not None
    x.add_row(inv.invoice_id, str(inv.invoice_date), f"{inv.total_kbd:,.2f}",
              trip["destination"] if trip else "Kestrel Bay",
              f"{trip['depart']} to {trip['return']}" if trip else "")
console.print(x)
note("The join is ordinary Python. The model only had to produce rows a program can trust.")
learn.reveal(p_trips, answer=0 if in_trips == 0 else (2 if in_trips == len(nw) else 1),
             why=f"{in_trips} of {len(nw)}. The model extracted; a date comparison in code made "
                 "the link, exactly and the same way every time.")

learn.record(
    no_schema_valid=no_schema_valid,
    invoices=len(invoices),
    valid_rows=len(rows),
    flagged=len(invoices) - len(rows),
    calls_total=attempts_used,
    retries_used=attempts_used - len(invoices),
    source_problems=problems,
    northwind_rows=len(nw),
    northwind_in_trips=in_trips,
    mine=bool(MINE),
)

your_turn(
    "08",
    check=[
        "Look at the first character of the no-schema reply. Why did validation reject it?",
        "Open case/invoices/NW-0419.txt and compare it with its row in the table. Is the vendor in "
        "normal capitalisation? Is every category the one you would pick?",
        "Read the step 3 notes. Which field had to be corrected, and what had the model put there?",
    ],
    exercises=[
        "In run.py, in `Invoice`, change `total_kbd` to plain `total_kbd: float` with no "
        "description. What type and format come back?",
        "In run.py, in extract(), change `format=schema` to `format=\"json\"` (JSON mode). "
        "Which fields drift?",
        "In run.py, move the `category` line to the top of `Invoice`, above `invoice_id`. "
        "Do the categories get worse?",
        "In run.py, change the `vat_number` description to \"Only the value printed after 'VAT "
        "registration:'. null if that line is absent.\" Does the source check still find problems?",
    ],
)

finding(f"""
{len(nw)} Northwind invoices, KBD {sum(i.total_kbd for i in nw):,.2f} in total. None carries a VAT
number. All of them remit to the same account at Sabran Mercantile Bank. And every one is dated
while Raj Malhotra was in Port Sabran on travel approved by Elena Voss.

Okafor pins a new card to the centre of his wall and draws three lines to it. "Operations does not
approve freight invoices," he says. "So why is every one of these coded to his cost centre?"
""")
save_shot("08-structured-output")

## The nuances

**1. Asking nicely fails in boring ways.** In our run the unconstrained reply began "Here is the
extracted invoice data in JSON format" and then a ```` ```json ```` fence. It also returned the total as
the string `"94,000.00"`, copied the service line into `category`, and put the vendor's email
address in `vat_number`. `json.loads` fails on the first character.

**2. Constrained is not correct.** With the schema, all eight invoices parsed and validated first
time. They still contained mistakes. The model filled `vat_number` for all three Northwind invoices
with the bank account number `88-301-4471`, although Northwind invoices carry no VAT line. A
nullable field gives the model permission to say null. It does not make it use that permission.
Step 3 of the script catches this by checking the field against the source text.

**3. Field order is generation order.** The model writes fields top to bottom, and each field is
context for the next. Put the fields that anchor the record first (`invoice_id`, `vendor`) and
derived judgements (`category`) after the facts they depend on. If you want reasoning, put a
`reasoning` field *before* the answer field, never after.

**4. Descriptions are prompts.** `Field(description=...)` text goes into the schema the model sees.
"Total due as a plain number, no commas or currency" is an instruction, and it is why `total_kbd`
came back as `94000.0` and not `"KBD 94,000.00"`.

**5. Enums turn free text into categories.** `Literal[...]` becomes a JSON schema `enum`. The model
can only pick from the list. It can still pick badly: Bayline IT's network switch replacement came
back as `equipment hire`, not `it`. Enums remove spelling variants, not judgement errors.

**6. Numbers, dates and nulls need types.** `"94,000.00"` is a string; `94000.0` is a number. Typed
fields (`float`, `date`, `str | None`) make the grammar enforce the type, and Pydantic convert it.
Never parse amounts out of prose with a regular expression if you can make the model emit a number.

**7. Validate, then retry with the error.** `extract()` feeds a failed validation message back to
the model as a new user turn and asks it to fix it. With constrained decoding the retry was never
needed here, but with "ask nicely" or a cloud model without schema support it is the standard
pattern. Cap the retries and send what still fails to a person.

**8. Let the model extract and let code reason.** The model only turns invoices into rows. The join
against Malhotra's travel dates is ordinary Python. It is exact, reproducible and cheap, and Marcus
Hale cannot argue with a date comparison.

## On a hosted model

```bash
make lesson N=08 TIER=hosted              # needs ANTHROPIC_API_KEY in .env, uv sync --extra cloud
make lesson N=08 TIER=hosted ARGS=--mine  # your extract()
make lesson N=08 TIER=hosted REPLAY=1     # once a hosted run is recorded (RECORD=1), no key
```

The hosted path uses Claude's **native structured outputs**: `format=schema` becomes
`output_config` with a strict JSON schema (`lantern.cloud.strict_schema`: every object closed,
every property required, nullable fields kept nullable). Step 1 sends no schema on purpose, as
locally. Claude takes no `temperature`, so the lesson's `temperature=0` is reported and not sent
(see `src/lantern/cloud.py`).

| Measure | Local: qwen3:1.7b | Hosted: claude-haiku-5-5 |
|---|---|---|
| No-schema reply passes validation | no | not run |
| Valid rows / calls, 8 invoices | 8 / 8 calls | not run |
| Fields contradicted by the source check | 3 (vat_number) | not run |
| Northwind invoices inside Malhotra's trips | 3 of 3 | not run |

Local numbers are from the screenshot's run (qwen3:1.7b, local tier, Apple M1 Pro 16 GB, 23 November 2026, temperature 0).

What the hosted run will test: shape failures should be gone on both, since both enforce the
schema. The bank account in `vat_number` and the capitalised vendor names are expected to be
small-model effects; the rule that a valid record still needs a check against the source holds
on any model, and the hosted run will show whether that check finds anything there.

## Try this

1. Remove the `description` from `total_kbd` and rerun. What type and format come back?
2. Change `format=schema` to `format="json"` (JSON mode). Which fields drift?
3. Move `category` to be the first field in `Invoice`. Do the categories get worse?
4. Change the `vat_number` description to "Only the value printed after 'VAT registration:'.
   null if that line is absent." Does the source check still find problems?

## Master it

After this chapter you can constrain a model's output with a schema, validate every record and
retry with the error under a cap, check valid records against the source, and keep the
reasoning in code.

**Predict, then see.** `make lesson N=08` stops four times to ask whether the unconstrained
reply validates, how many calls eight invoices take, how many fields the source check flags,
and how many Northwind invoices fall inside Malhotra's trips. Each answer is checked against
your own run. Add `ARGS=--no-predict` to watch without stopping.

**Build it.** Write `extract()` in [`practice/exercise.py`](practice/exercise.py): the
schema-constrained call, Pydantic validation, and a retry that shows the model its validation
error, with an attempt cap. The eval suite in chapter 18 makes the same constrained call.

```bash
make exercise N=08             # 7 offline tests, scripted model, no Ollama needed
make hint N=08                 # one hint at a time
make solution N=08             # the reference, once you have tried
make lesson N=08 ARGS=--mine   # all eight invoices through your extract()
```

**Check yourself.** `make quiz N=08`: seven questions, each with the reason behind the answer.

**Break it.** `make challenge N=08` checks your latest run against three targets: your
extract() giving eight valid rows; a validator the grammar cannot enforce, so the retry has to
run; and field descriptions that leave the source check with nothing to flag.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 08

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 08

In [ ]:
%run ../../scripts/learn.py hint 08

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 08

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

Three Northwind invoices, KBD 227,700.00 in total. None carries a VAT number. All of them remit to
the same account at Sabran Mercantile Bank. And every one is dated while Raj Malhotra was in Port
Sabran on travel approved by Elena Voss.

Okafor pins a new card to the centre of his wall and draws three lines to it. "Operations does not
approve freight invoices," he says. "So why is every one of these coded to his cost centre?"

**Next:** [Chapter 09: Sixteen minutes](../09-chain-of-thought/README.md)